# 1일차 실습 P1 — 크기 먼저 적고 오토인코더 쓰기

- 수업 중 이 실습 슬라이드가 나오면 풂 · **맨 위 준비 셀부터** 위에서 아래로 실행
- 셀 실행: 셀을 누르고 **Shift + Enter** 또는 셀 왼쪽 ▶
- Colab 에서 처음 열 때 경고 창이 뜨면 '계속' · 새 파일은 준비 셀에서 데이터를 받느라 몇십 초 걸릴 수 있음
- 빈칸은 `____` · 빈칸을 모두 바꾼 뒤 **그 셀부터 다시 실행**
- 막히면 셀 이름과 **에러 칸 맨 아래 줄**을 채팅으로

## 에러를 읽는 법

- 에러 칸 첫 줄(`...Error    Traceback ...`)은 제목일 뿐 · 무엇이 틀렸는지는 **맨 아래 줄**
- 중간의 `---->` 화살표 줄 · torch 안쪽 칸은 처음엔 건너뜀 · **위에 찍힌 `[안내]` 문장과 맨 아래 줄부터** 읽음

| 맨 아래 줄에 보이는 말 | 뜻 | 먼저 볼 곳 |
|---|---|---|
| `name '____' is not defined` | 빈칸이 남음 | 화살표가 가리키는 줄 |
| `name 'nn'` · `'Xtr'` · `'MyAE'` · `'MyConvAE'` · `'크기_검사'` is not defined | 그 이름을 만든 셀을 안 돌렸거나 런타임이 끊김 | 준비 셀부터 순서대로 다시 |
| `name 'latent_dimm'` · `'Sigmoid'` · `'optimizer'` · `'ae'` is not defined | 내가 친 이름이 틀림 · 철자 · `nn.` 빠짐 · 학습 함수 안 이름은 `model` · `opt` | 화살표가 가리키는 줄 |
| `unexpected indent` · `unindent` | 줄 앞 칸 수가 위아래 줄과 다름 | 바로 위 줄과 같은 칸에서 시작하게 |
| `Perhaps you forgot a comma?` · `'(' was never closed` · `unmatched ')'` | 쉼표나 괄호 | `^` 표시 자리 |
| `mat1 and mat2 shapes cannot be multiplied (AxB and CxD)` | Linear 에 들어온 값 개수 B 와 Linear 첫 숫자 C 가 다름 | `Flatten` 과 `Linear` 첫 숫자 |
| `is not a Module subclass` | 부품 뒤 `()` 가 빠짐 | `nn.ReLU` → `nn.ReLU()` |
| `expected input[...] to have a channels, but got c channels` | 층의 첫 숫자(받는 채널)가 들어온 채널과 다름 | 마지막으로 찍힌 층의 **다음** 층 첫 숫자 |
| `The size of tensor a (..) must match the size of tensor b (..)` | 출력 크기가 정답 크기와 다름 | 층마다 찍힌 모양 · 이름의 대소문자(`x` · `X`) |

- **에러가 없는데** loss 가 0.2 근처에서 안 내려가고 그림이 회색 네모 → 학습이 안 됐을 수 있음 · `backward()` · `step()` 괄호부터 확인

## 준비

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets

tr = datasets.MNIST("data", train=True, download=True)
te = datasets.MNIST("data", train=False, download=True)
Xtr = tr.data.float().div(255).unsqueeze(1)   # 학습 60,000장 · 값 0~1
Xte = te.data.float().div(255).unsqueeze(1)   # 평가 10,000장
SHOW = [5459, 9898, 926, 1150, 5691, 2400, 7792, 950]   # 평가 이미지 중 숫자 0~7 한 장씩


@torch.no_grad()
def run(model, X):                      # 2,000장씩 넣어 복원을 모음
    model.eval()
    return torch.cat([model(X[i:i + 2000]) for i in range(0, len(X), 2000)])


def show(rows):                         # 같은 여덟 장을 줄마다 나란히
    fig, axes = plt.subplots(len(rows), 8, figsize=(8, 1.15 * len(rows)))
    for r, (name, X) in enumerate(rows):
        for c in range(8):
            axes[r, c].imshow(X[SHOW[c], 0], cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()


print("준비 끝 · 학습", tuple(Xtr.shape), " 평가", tuple(Xte.shape))

100.0%
100.0%
100.0%
100.0%

준비 끝 · 학습 (60000, 1, 28, 28)  평가 (10000, 1, 28, 28)


## P1. 층마다 크기 먼저 적고 오토인코더 쓰기

1. `LATENT` 에 잠재 크기를 하나 정함(예: 16 · 8 · 4)
2. **코드보다 먼저** 아래 표의 빈칸(나오는 모양)을 채움 · 입력 줄처럼 괄호 모양으로 · 표는 더블클릭해서 고치거나 편집이 어려우면 종이 · 채팅에
3. 인코더 · 디코더 뼈대의 빈칸에 숫자와 마지막 함수 이름만 채움 → 확인 셀 실행 → 적어 둔 모양과 찍힌 모양 비교

| 자리 | 한 장 넣었을 때 나오는 모양 |
|---|---|
| 입력 | (1, 1, 28, 28) |
| `nn.Flatten()` 뒤 | (1, 784) |
| 인코더 끝 · 잠재 벡터 | (1, 16) |
| 디코더 끝을 `reshape` 한 복원 | (1, 784) ❌ | 
| 디코더 끝을 `reshape` 한 복원 | (1, 1, 28, 28) ✅| 

In [ ]:
LATENT = 16   # 잠재 크기 하나 (예: 16) · 아래 확인 셀이 MyAE(LATENT) 로 만들어 이 값이 클래스 안 latent_dim 으로 들어감

# 빈칸: nn.Linear(받는 개수, 내놓는 개수) 의 숫자 · 마지막 함수 이름
# 예: nn.Linear(3, 5) = 값 3개를 받아 5개를 내놓음 · 쉼표 · 괄호는 이미 적혀 있음


class MyAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(          # 줄이기: 784 → 128 → latent_dim
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 16))
        self.decoder = nn.Sequential(          # 되돌리기: latent_dim → 128 → 784 · 끝은 0~1 로
            nn.Linear(16, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid())                       # Why Sigmoid? -> 입력 범위와 맞추려고 

    def forward(self, x):
        return self.decoder(self.encoder(x)).reshape(-1, 1, 28, 28)

### 막히면 — 힌트 1 (숫자 자리)

- 인코더: 첫 `nn.Linear` 는 784 → 128 · 둘째 `nn.Linear` 는 128 → latent_dim
- 디코더: 첫 `nn.Linear` 는 latent_dim → 128 · 둘째 `nn.Linear` 는 128 → 784
- `latent_dim` 은 변수 이름 · 빈칸에 `latent_dim` 이라고 이름 그대로 써도 됨

### 그래도 막히면 — 힌트 2 (마지막 함수)

- 디코더 끝 함수는 픽셀마다 값을 0~1 로 만드는 것 · 이름은 `Sigmoid` (첫 글자 대문자)

In [3]:
# P1 확인 — 적어 둔 모양과 비교
torch.manual_seed(0)
x = Xte[:1]
try:
    my_ae = MyAE(LATENT)
    print("입력       ", tuple(x.shape))
    print("Flatten 뒤 ", tuple(my_ae.encoder[0](x).shape))
    print("잠재 벡터  ", tuple(my_ae.encoder(x).shape))
    print("복원       ", tuple(my_ae(x).shape))
    if my_ae.encoder(x).shape[1] != LATENT:
        print("[안내] 잠재 벡터 크기가 LATENT 와 다름 → 인코더 둘째 Linear 의 내놓는 개수는 latent_dim")
    if not isinstance(my_ae.decoder[-1], nn.Sigmoid):
        print("[안내] 디코더 끝이 nn.Sigmoid() 가 아님 → 픽셀마다 0~1 로 만드는 함수")
except Exception as e:
    msg = str(e)
    if "____" in msg:
        print("[안내] 빈칸 ____ 이 남아 있음 → 위 셀의 ____ 를 모두 바꾸고 위 셀부터 다시 실행")
    elif "'MyAE' is not defined" in msg or "'LATENT' is not defined" in msg:
        print("[안내] 위 P1 셀이 실행되지 않았거나 에러로 멈춤 → 위 셀의 빈칸을 모두 바꾸고 위 셀부터 다시 실행")
    elif "mat1 and mat2" in msg:
        print("[안내] Linear 에 들어온 값 개수와 Linear 첫 숫자가 다름 · (AxB and CxD) 에서 B 가 들어온 개수, C 가 Linear 첫 숫자")
    elif "not a Module subclass" in msg:
        print("[안내] 부품 이름 뒤 () 가 빠짐 · 예: nn.ReLU → nn.ReLU()")
    elif "not callable" in msg:
        print("[안내] nn.Sequential( ... ) 로 감싸지 않았음")
    elif "is invalid for input of size" in msg:
        print("[안내] 디코더 끝 Linear 가 784 를 내놓지 않음")
    elif "has no attribute" in msg:
        print("[안내] nn. 뒤 함수 이름의 철자 · 대문자 확인 · 예: nn.sigmoid → nn.Sigmoid")
    elif "is not defined" in msg:
        print("[안내] 내가 친 이름을 찾지 못함 → 철자 확인 · 잠재 크기는 latent_dim · 부품은 nn.Sigmoid() 처럼 nn. 을 붙임")
    raise

입력        (1, 1, 28, 28)
Flatten 뒤  (1, 784)
잠재 벡터   (1, 16)
복원        (1, 1, 28, 28)
